# EMT SSN GFM load-step simulation

This notebook is the Python/pybind equivalent of the C++ example `EMT_SSN_GFM_LoadStep`.
Parametrization of the GFM and the used GFM model is derived from X. Gao, D. Zhou, A. Anvari-Moghaddam and F. Blaabjerg, "Stability Analysis of Grid-Following and Grid-Forming Converters Based on State-Space Model" 


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

import dpsimpy

emt = dpsimpy.emt
sp = dpsimpy.sp

ph3 = emt.ph3
sp_ph1 = sp.ph1

Simulation = dpsimpy.Simulation
SystemTopology = dpsimpy.SystemTopology
Logger = dpsimpy.Logger
Domain = dpsimpy.Domain
PhaseType = dpsimpy.PhaseType
PowerflowBusType = dpsimpy.PowerflowBusType
Solver = dpsimpy.Solver
SolverBehaviour = dpsimpy.SolverBehaviour
SwitchEvent3Ph = dpsimpy.event.SwitchEvent3Ph

## Simulation and electrical parameters


In [ ]:
final_time = 3.0
time_step = 100e-6

sim_name = "EMT_SSN_GFM_LoadStep"
sim_name_pf = sim_name + "_PF"
sim_name_emt = sim_name + "_EMT"

phase_voltage_rms = 220.0
line_to_line_voltage_rms = np.sqrt(3.0) * phase_voltage_rms
phase_voltage_peak = np.sqrt(2.0) * phase_voltage_rms

nominal_frequency = 50.0
nominal_omega = 2.0 * np.pi * nominal_frequency

filter_inductance = 3e-3
filter_capacitance = 20e-6
filter_resistance = 0.05
internal_coupling_resistance = 0.05

connection_resistance = 0.20

virtual_inertia = 0.2
damping_coefficient = 25.0
voltage_droop_gain = 1.0 / 15.0

voltage_loop_bandwidth = 200.0
current_loop_bandwidth = 1683.0
power_filter_cutoff = 100.0
sampling_frequency = 20e3

## Controller gains and operating point


In [ ]:
voltage_loop_damping = 1.0 / np.sqrt(2.0)

kp_voltage = 2.0 * voltage_loop_damping * voltage_loop_bandwidth * filter_capacitance

ki_voltage = voltage_loop_bandwidth**2 * filter_capacitance

kp_current = current_loop_bandwidth * filter_inductance
ki_current = current_loop_bandwidth * filter_resistance

reactive_integral_gain = 300.0 / 1000.0

active_damping_gain = 0.0

delay_bandwidth = sampling_frequency / 1.5

base_load_active_power = 12e3
base_load_reactive_power = 0.0

step_load_active_power = 3e3
step_load_reactive_power = 0.0

active_power_reference = base_load_active_power
reactive_power_reference = base_load_reactive_power

load_step_start_time = 1.0
load_step_end_time = 2.0

switch_open_resistance = 1e9
switch_closed_resistance = 1e-6

## Helper functions


In [ ]:
def three_phase_parameter(value):
    return dpsimpy.Math.single_phase_parameter_to_three_phase(value)


def three_phase_power(value):
    return dpsimpy.Math.single_phase_power_to_three_phase(value)


def log_attr(logger, name, obj, attr_name):
    if hasattr(obj, "attr"):
        logger.log_attribute(name, obj.attr(attr_name))
    else:
        logger.log_attribute(name, obj.attribute(attr_name))


def load_csv(name):
    path = Path("logs") / name / f"{name}.csv"
    return pd.read_csv(path, skipinitialspace=True)


def signal_columns(df, prefix):
    return [
        column
        for column in df.columns
        if column == prefix or column.startswith(prefix + "_")
    ]

# Power-flow initialization


In [ ]:
def build_and_run_powerflow():
    Logger.set_log_dir("logs/" + sim_name_pf)

    time_step_pf = 1.0
    final_time_pf = 2.0

    n_gfm_pf = sp.SimNode("nGfm", PhaseType.Single)
    n_load_pf = sp.SimNode("nLoad", PhaseType.Single)

    extnet_pf = sp_ph1.NetworkInjection("Slack")
    extnet_pf.set_parameters(line_to_line_voltage_rms)
    extnet_pf.set_base_voltage(line_to_line_voltage_rms)
    extnet_pf.modify_power_flow_bus_type(PowerflowBusType.VD)

    connection_pf = sp_ph1.PiLine("Connection")
    connection_pf.set_parameters(
        connection_resistance,
        0.0,
        0.0,
    )
    connection_pf.set_base_voltage(line_to_line_voltage_rms)

    base_load_pf = sp_ph1.Load("BaseLoad")
    base_load_pf.set_parameters(
        base_load_active_power,
        base_load_reactive_power,
        line_to_line_voltage_rms,
    )

    extnet_pf.connect([n_gfm_pf])
    connection_pf.connect([n_gfm_pf, n_load_pf])
    base_load_pf.connect([n_load_pf])

    system_pf = SystemTopology(
        nominal_frequency,
        [n_gfm_pf, n_load_pf],
        [extnet_pf, connection_pf, base_load_pf],
    )

    logger_pf = Logger(sim_name_pf)
    log_attr(logger_pf, "Voltage_GFM", n_gfm_pf, "v")
    log_attr(logger_pf, "Voltage_Load", n_load_pf, "v")

    sim_pf = Simulation(sim_name_pf)
    sim_pf.set_system(system_pf)
    sim_pf.set_time_step(time_step_pf)
    sim_pf.set_final_time(final_time_pf)
    sim_pf.set_domain(Domain.SP)
    sim_pf.set_solver(Solver.NRP)
    sim_pf.set_solver_component_behaviour(SolverBehaviour.Initialization)
    sim_pf.do_init_from_nodes_and_terminals(False)
    sim_pf.add_logger(logger_pf)
    sim_pf.run()

    return system_pf


system_pf = build_and_run_powerflow()

# EMT simulation


In [ ]:
def build_and_run_emt(system_pf):
    Logger.set_log_dir("logs/" + sim_name_emt)

    n_gfm_emt = emt.SimNode("nGfm", PhaseType.ABC)
    n_load_emt = emt.SimNode("nLoad", PhaseType.ABC)
    n_step_load_emt = emt.SimNode("nStepLoad", PhaseType.ABC)

    gfm = ph3.SSN_GFM(
        "GFM",
        "GFM",
    )

    gfm.set_parameters(
        filter_inductance,
        filter_capacitance,
        filter_resistance,
        internal_coupling_resistance,
        phase_voltage_peak,
        nominal_omega,
        active_power_reference,
        reactive_power_reference,
        virtual_inertia,
        damping_coefficient,
        voltage_droop_gain,
        reactive_integral_gain,
        kp_voltage,
        ki_voltage,
        kp_current,
        ki_current,
        active_damping_gain,
        power_filter_cutoff,
        delay_bandwidth,
    )

    gfm.set_numerical_linearization_parameters(
        1e-6,
        1e-8,
    )

    connection_emt = ph3.Resistor("Connection")
    connection_emt.set_parameters(three_phase_parameter(connection_resistance))

    base_load_emt = ph3.RXLoad("BaseLoad")
    base_load_emt.set_parameters(
        three_phase_power(base_load_active_power),
        three_phase_power(base_load_reactive_power),
        line_to_line_voltage_rms,
    )

    step_load_emt = ph3.RXLoad("StepLoad")
    step_load_emt.set_parameters(
        three_phase_power(step_load_active_power),
        three_phase_power(step_load_reactive_power),
        line_to_line_voltage_rms,
    )

    load_switch = ph3.Switch("LoadSwitch")
    load_switch.set_parameters(
        three_phase_parameter(switch_open_resistance),
        three_phase_parameter(switch_closed_resistance),
    )
    load_switch.open()

    gfm.connect([emt.SimNode.gnd, n_gfm_emt])

    connection_emt.connect([n_gfm_emt, n_load_emt])
    base_load_emt.connect([n_load_emt])
    load_switch.connect([n_load_emt, n_step_load_emt])
    step_load_emt.connect([n_step_load_emt])

    system_emt = SystemTopology(
        nominal_frequency,
        [n_gfm_emt, n_load_emt, n_step_load_emt],
        [
            gfm,
            connection_emt,
            base_load_emt,
            load_switch,
            step_load_emt,
        ],
    )

    system_emt.init_with_powerflow(system_pf, Domain.EMT)

    logger_emt = Logger(sim_name_emt)

    log_attr(logger_emt, "Voltage_GFM", n_gfm_emt, "v")
    log_attr(logger_emt, "Voltage_Load", n_load_emt, "v")
    log_attr(logger_emt, "Voltage_StepLoad", n_step_load_emt, "v")
    log_attr(logger_emt, "GFM_InterfaceVoltage", gfm, "v_intf")
    log_attr(logger_emt, "GFM_InterfaceCurrent", gfm, "i_intf")
    log_attr(logger_emt, "GFM_ActivePower", gfm, "p_inst")
    log_attr(logger_emt, "GFM_ReactivePower", gfm, "q_inst")
    log_attr(logger_emt, "GFM_Omega", gfm, "omega_gfm")
    log_attr(logger_emt, "GFM_Theta", gfm, "theta_gfm")
    log_attr(
        logger_emt,
        "GFM_VoltageMagnitude",
        gfm,
        "voltage_magnitude_gfm",
    )

    log_attr(logger_emt, "GFM_CapacitorVoltageD", gfm, "vc_d")
    log_attr(logger_emt, "GFM_CapacitorVoltageQ", gfm, "vc_q")
    log_attr(logger_emt, "GFM_GridCurrentD", gfm, "i_grid_d")
    log_attr(logger_emt, "GFM_GridCurrentQ", gfm, "i_grid_q")
    log_attr(logger_emt, "GFM_FilterCurrentD", gfm, "if_d")
    log_attr(logger_emt, "GFM_FilterCurrentQ", gfm, "if_q")

    sim = Simulation(sim_name_emt)

    sim.add_event(
        SwitchEvent3Ph(
            load_step_start_time,
            load_switch,
            True,
        )
    )
    sim.add_event(
        SwitchEvent3Ph(
            load_step_end_time,
            load_switch,
            False,
        )
    )

    sim.set_system(system_emt)
    sim.set_time_step(time_step)
    sim.set_final_time(final_time)
    sim.set_domain(Domain.EMT)
    sim.set_solver(Solver.MNA)
    sim.do_system_matrix_recomputation(True)
    sim.do_init_from_nodes_and_terminals(True)
    sim.add_logger(logger_emt)
    sim.run()

    return system_emt, gfm


system_emt, gfm = build_and_run_emt(system_pf)

# Load simulation results


In [ ]:
pf_result = load_csv(sim_name_pf)
emt_result = load_csv(sim_name_emt)

print("PF result:")
display(pf_result.head())

print("EMT result:")
display(emt_result.head())

# Inspect the GFM state through pybind


In [ ]:
state_names = gfm.get_state_names()
state_values = np.asarray(gfm.get_state()).reshape(-1)

state_result = pd.DataFrame(
    {
        "state": state_names,
        "value": state_values,
    }
)

# Plot active and reactive power


In [ ]:
time_column = emt_result.columns[0]
time = emt_result[time_column].to_numpy()

for prefix, ylabel, title in [
    ("GFM_ActivePower", "P [W]", "GFM active power"),
    ("GFM_ReactivePower", "Q [var]", "GFM reactive power"),
]:
    columns = signal_columns(emt_result, prefix)

    if not columns:
        print(f"Skipping {prefix}: no columns found.")
        continue

    plt.figure(figsize=(12, 5))

    for column in columns:
        plt.plot(time, emt_result[column], label=column)

    plt.axvline(
        load_step_start_time,
        linestyle="--",
        label="Load step connected",
    )
    plt.axvline(
        load_step_end_time,
        linestyle="--",
        label="Load step disconnected",
    )

    plt.xlabel("Time [s]")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True)
    plt.legend(loc="best")
    plt.tight_layout()
    plt.show()

# Plot GFM frequency


In [ ]:
omega_columns = signal_columns(emt_result, "GFM_Omega")

if omega_columns:
    plt.figure(figsize=(12, 5))

    for column in omega_columns:
        frequency = emt_result[column] / (2.0 * np.pi)
        plt.plot(time, frequency, label=column)

    plt.axhline(
        nominal_frequency,
        linestyle="--",
        label="Nominal frequency",
    )
    plt.axvline(load_step_start_time, linestyle="--")
    plt.axvline(load_step_end_time, linestyle="--")

    plt.xlabel("Time [s]")
    plt.ylabel("Frequency [Hz]")
    plt.title("GFM frequency")
    plt.grid(True)
    plt.legend(loc="best")
    plt.tight_layout()
    plt.show()
else:
    print("No GFM omega column found.")

# Plot dq voltages and currents


In [ ]:
for signals, ylabel, title in [
    (
        [
            "GFM_CapacitorVoltageD",
            "GFM_CapacitorVoltageQ",
            "GFM_VoltageMagnitude",
        ],
        "Voltage [V]",
        "GFM dq voltages",
    ),
    (
        [
            "GFM_GridCurrentD",
            "GFM_GridCurrentQ",
            "GFM_FilterCurrentD",
            "GFM_FilterCurrentQ",
        ],
        "Current [A]",
        "GFM dq currents",
    ),
]:
    available_columns = [
        column for signal in signals for column in signal_columns(emt_result, signal)
    ]

    if not available_columns:
        print(f"Skipping {title}: no columns found.")
        continue

    plt.figure(figsize=(12, 5))

    for column in available_columns:
        plt.plot(time, emt_result[column], label=column)

    plt.axvline(load_step_start_time, linestyle="--")
    plt.axvline(load_step_end_time, linestyle="--")

    plt.xlabel("Time [s]")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True)
    plt.legend(loc="best")
    plt.tight_layout()
    plt.show()

# Plot three-phase node voltages


In [ ]:
voltage_signals = [
    ("GFM node", "Voltage_GFM"),
    ("Base-load node", "Voltage_Load"),
    ("Step-load node", "Voltage_StepLoad"),
]

for title, prefix in voltage_signals:
    columns = signal_columns(emt_result, prefix)

    if not columns:
        print(f"No columns found for {title}.")
        continue

    plt.figure(figsize=(12, 5))

    for column in columns:
        plt.plot(time, emt_result[column], label=column)

    plt.axvline(load_step_start_time, linestyle="--")
    plt.axvline(load_step_end_time, linestyle="--")

    plt.xlabel("Time [s]")
    plt.ylabel("Voltage [V]")
    plt.title(title)
    plt.grid(True)
    plt.legend(loc="best")
    plt.tight_layout()
    plt.show()

# Three-phase voltage magnitude


In [ ]:
for title, prefix in voltage_signals:
    columns = signal_columns(emt_result, prefix)

    if len(columns) != 3:
        print(f"{title}: expected three phase columns, " f"found {len(columns)}.")
        continue

    va = emt_result[columns[0]]
    vb = emt_result[columns[1]]
    vc = emt_result[columns[2]]

    voltage_magnitude = np.sqrt(2.0 / 3.0 * (va**2 + vb**2 + vc**2))

    plt.figure(figsize=(12, 5))
    plt.plot(
        time,
        voltage_magnitude,
        label="Voltage magnitude",
    )

    plt.axhline(
        phase_voltage_peak,
        linestyle="--",
        label=f"Reference: {phase_voltage_peak:.2f} V",
    )
    plt.axvline(load_step_start_time, linestyle="--")
    plt.axvline(load_step_end_time, linestyle="--")

    plt.xlabel("Time [s]")
    plt.ylabel("Voltage magnitude [V]")
    plt.title(title)
    plt.grid(True)
    plt.legend(loc="best")
    plt.tight_layout()
    plt.show()